# QLoRA Fine-tuning Qwen/Qwen2.5-7B-Instruct: Corporate Clerk Tone
Production-grade pipeline testing LoRA ranks with MLflow tracking and vLLM inference.

In [ ]:
# Cell 1: Dependencies & MLflow Setup
!pip install -q -U vllm mlflow trl peft bitsandbytes datasets transformers accelerate matplotlib

import mlflow
import os

# Set up local MLflow tracking
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlruns.db"
mlflow.set_experiment("Qwen_Clerk_Summarization")
print("MLflow tracking initialized and experiment set.")

In [ ]:
# Cell 2: Dataset Generation
import json

dummy_data = [
    {"raw_text": "Hey so I was talking to the marketing guys and they said the new campaign is totally busted and we are bleeding money.", "clerk_summary": "The marketing department reports that the current campaign is underperforming, resulting in adverse financial variance."},
    {"raw_text": "John wants to take off like two weeks because his dog is sick, can we just let him do it?", "clerk_summary": "An employee request for two weeks of extended leave due to pet illness is pending evaluation under the current HR protocols."},
    {"raw_text": "The servers crashed again. IT is running around like headless chickens but honestly it's AWS's fault.", "clerk_summary": "We are experiencing an ongoing server outage. The IT department is currently liaising with our cloud provider to mitigate the external service disruption."},
    {"raw_text": "Dude the client is super mad, he wants his money back right now.", "clerk_summary": "A client has formally expressed dissatisfaction and is requesting an immediate financial reimbursement."},
    {"raw_text": "Can someone please buy more coffee for the breakroom? It's basically an emergency at this point.", "clerk_summary": "There is a pending supply requisition for breakroom continuous provisions."},
    {"raw_text": "We need to delay the launch to next Tuesday, the devs found a huge bug.", "clerk_summary": "The product deployment schedule must be amended to the following Tuesday due to a critical defect identified by the engineering division."},
    {"raw_text": "I can't log in to my account, the password reset thing is broken.", "clerk_summary": "An access impediment has been documented relating to the credential reset protocol."},
    {"raw_text": "The new hire is pretty much useless and doesn't know how to code.", "clerk_summary": "Preliminary evaluations indicate that the recent hire may require additional training or reassignment due to misaligned technical proficiencies."},
    {"raw_text": "Who approved this expense report? It's like $500 for a team lunch.", "clerk_summary": "There is a $500 team meal expenditure under review requiring clarification of the approval chain authorization."},
    {"raw_text": "We ran out of printer ink again, tell facilities to get on it.", "clerk_summary": "A facilities request has been formally submitted to replenish printer consumable inventory."}
]

jsonl_file = "raw_to_clerk.jsonl"
with open(jsonl_file, "w", encoding="utf-8") as f:
    for item in dummy_data:
        f.write(json.dumps(item) + "\n")

print(f"Synthetic dataset generated at {jsonl_file} with {len(dummy_data)} samples.")

Generated dataset at corporate_tone.jsonl with 10 samples.


In [ ]:
# Cell 3: Formatting & Tokenization
from transformers import AutoTokenizer
from datasets import load_dataset

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure padding configuration is correct for causal LM
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("json", data_files="raw_to_clerk.jsonl", split="train")

def format_chat_template(example):
    system_prompt = "You are a Corporate Clerk. Summarize the following informal text into a highly formal, dry, bureaucratic summary."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example["raw_text"]},
        {"role": "assistant", "content": example["clerk_summary"]}
    ]
    # Apply standard model's native chat template
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": formatted_text}

formatted_dataset = dataset.map(format_chat_template)
print("Preview of formatted dataset:\n", formatted_dataset[0]["text"])

Loading Tokenizer from Qwen/Qwen2.5-3B-Instruct...
Loading Model Qwen/Qwen2.5-3B-Instruct in 4-bit...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
# Cell 4: Hyperparameter Sweep & Training (The MLflow Experiment)
import gc
import torch
import mlflow
import itertools
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Split the dataset to allow evaluation during the hyperparameter sweep
split_dataset = formatted_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

ranks = [8, 16, 32]
learning_rates = [1e-4, 5e-5, 1e-5]

for current_rank, current_lr in itertools.product(ranks, learning_rates):
    lora_alpha = 2 * current_rank
    print(f"\n--- Starting run for r={current_rank}, lr={current_lr} ---")
    
    with mlflow.start_run(run_name=f"LoRA_r{current_rank}_lr{current_lr}") as run:
        mlflow.log_params({
            "lora_rank": current_rank,
            "learning_rate": current_lr,
            "lora_alpha": lora_alpha
        })
        
        # 1. Load 4-bit model inside the loop for memory safety
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = prepare_model_for_kbit_training(model)
        
        # 2. Configure PEFT (LoRA)
        peft_config = LoraConfig(
            r=current_rank,
            lora_alpha=lora_alpha,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
        )
        model = get_peft_model(model, peft_config)
        
        # 3. Training arguments
        training_args = SFTConfig(
            output_dir=f"./lora_r{current_rank}_lr{current_lr}",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=30, # Quick testing
            learning_rate=current_lr,
            optim="paged_adamw_8bit",
            bf16=True,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=10,
            report_to="mlflow",
            remove_unused_columns=True, 
            max_length=512,             
            dataset_text_field="text"   
        )
        
        trainer = SFTTrainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            args=training_args
        )
        
        # 4. Train
        trainer.train()
        
        # 5. Save adapter using dynamic folder naming
        trainer.model.save_pretrained(f"./lora_r{current_rank}_lr{current_lr}_final")
        tokenizer.save_pretrained(f"./lora_r{current_rank}_lr{current_lr}_final")
        
        # 6. CRITICAL VRAM MANAGEMENT
        print(f"Cleaning up VRAM for r={current_rank}, lr={current_lr}...")
        del model
        del trainer
        gc.collect()
        torch.cuda.empty_cache()
        print(f"Run for r={current_rank}, lr={current_lr} completed successfully.")

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Preview of the first formatted sample:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Hey, write an email telling John he's fired.<|im_end|>
<|im_start|>assistant
Subject: Important Update Regarding Your Employment

Dear John,

I am writing to inform you that your employment with the company will be terminated, effective immediately. Please contact Human Resources for details regarding your offboarding process.

Sincerely,
Management<|im_end|>



In [ ]:
# Cell 5: Advanced MLflow Analytics & Visualization
import mlflow
import matplotlib.pyplot as plt

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("Qwen_Clerk_Summarization")

if experiment:
    runs = client.search_runs(experiment.experiment_id)
    
    run_metrics = []
    for run in runs:
        run_id = run.info.run_id
        r_val = run.data.params.get('lora_rank', 'Unknown')
        lr_val = run.data.params.get('learning_rate', 'Unknown')
        
        # Extract eval_loss metrics correctly via the client 
        eval_metrics = client.get_metric_history(run_id, "eval/loss")
        if eval_metrics:
            # Capture the very last eval_loss value
            final_eval_loss = eval_metrics[-1].value
            run_metrics.append({
                "run_id": run_id,
                "r": r_val,
                "lr": lr_val,
                "final_eval_loss": final_eval_loss,
                "label": f"r={r_val}, lr={lr_val}"
            })
    
    if not run_metrics:
        print("No evaluation metrics found. Ensure that evaluation ran correctly.")
    else:
        # Sort runs starting with the lowest validation loss
        run_metrics.sort(key=lambda x: x['final_eval_loss'])
        
        # Plot 1: Bar Chart of all final validation losses
        labels = [x['label'] for x in run_metrics]
        losses = [x['final_eval_loss'] for x in run_metrics]
        
        plt.figure(figsize=(10, 6))
        plt.bar(labels, losses, color='steelblue')
        plt.title("Final Validation Loss across Configurations")
        plt.xlabel("Hyperparameters (Rank, Learning Rate)")
        plt.ylabel("Validation Loss")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        # Plot 2: Line Chart for Top 3 Runs over Time
        top_3_runs = run_metrics[:3]
        plt.figure(figsize=(10, 6))
        
        colors = ['blue', 'orange', 'green']
        for idx, run_info in enumerate(top_3_runs):
            run_id = run_info['run_id']
            label = run_info['label']
            color = colors[idx % len(colors)]
            
            # Fetch structured history
            train_hist = client.get_metric_history(run_id, "train/loss")
            train_steps = [m.step for m in train_hist]
            train_losses = [m.value for m in train_hist]
            
            eval_hist = client.get_metric_history(run_id, "eval/loss")
            eval_steps = [m.step for m in eval_hist]
            eval_losses = [m.value for m in eval_hist]
            
            # Overlay Solid lines for Train, Dashed lines for Eval
            plt.plot(train_steps, train_losses, color=color, linestyle='-', label=f"Train - {label}")
            plt.plot(eval_steps, eval_losses, color=color, linestyle='--', marker='o', label=f"Eval - {label}")
            
        plt.title("Top 3 Configurations: Training vs Validation Loss")
        plt.xlabel("Steps")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
else:
    print("Experiment not found. Check tracking setup.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss


Model saved to ./corporate_lora_final


In [ ]:
# Cell 6: VRAM Cleanup
import gc
import torch

# Mandatory aggressive cleanup before dynamic vLLM inference to free KV cache blocks
gc.collect()
torch.cuda.empty_cache()
print("PyTorch memory flushed. Ready for inference allocation.")

trainer params: ['self', 'model', 'args', 'data_collator', 'train_dataset', 'eval_dataset', 'processing_class', 'compute_loss_func', 'compute_metrics', 'callbacks', 'optimizers', 'optimizer_cls_and_kwargs', 'preprocess_logits_for_metrics', 'peft_config', 'formatting_func']
config params: ['self', 'output_dir', 'per_device_train_batch_size', 'num_train_epochs', 'max_steps', 'learning_rate', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_steps', 'optim', 'optim_args', 'weight_decay', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'optim_target_modules', 'gradient_accumulation_steps', 'average_tokens_across_devices', 'max_grad_norm', 'label_smoothing_factor', 'bf16', 'fp16', 'bf16_full_eval', 'fp16_full_eval', 'tf32', 'gradient_checkpointing', 'gradient_checkpointing_kwargs', 'torch_compile', 'torch_compile_backend', 'torch_compile_mode', 'use_liger_kernel', 'liger_kernel_config', 'use_cache', 'neftune_noise_alpha', 'torch_empty_cache_steps', 'auto_find_batch_size', 'logging_strategy', 

In [ ]:
# Cell 7: vLLM Dynamic Inference
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# Initialize vLLM with LoRA enabled and restrictive GPU util to fit in Google Colab T4
llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct",
    enable_lora=True,
    max_lora_rank=16,
    gpu_memory_utilization=0.7 # strictly 70% to avoid OOM
)

system_prompt = "You are a Corporate Clerk. Summarize the following informal text into a highly formal, dry, bureaucratic summary."
user_input = "We literally have no idea what we're doing and the servers are melting, pls help."
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_input}
]

# Note: We must use the tokenizer to construct the chat template
# as vllm accepts raw strings or prompt structures. We assume tokenizer is still in memory from Cell 3.
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

sampling_params = SamplingParams(temperature=0.3, max_tokens=150)

print("--- Base Model Output ---")
base_outputs = llm.generate(prompt, sampling_params)
print(base_outputs[0].outputs[0].text)

print("\n--- LoRA (r=16) Model Output ---")
lora_outputs = llm.generate(
    prompt, 
    sampling_params,
    # Point inference purely to our specific adapter
    lora_request=LoRARequest("clerk_adapter", 1, "./lora_r16_final")
)
print(lora_outputs[0].outputs[0].text)